# Perfilado reproducible · Spotify + Grammy
Requisitos AR1–AR3: cobertura, popularidad y perfil de audio. Leer docs/01_requirements.md antes de ejecutar. Este notebook no limpia ni modifica fuentes. Grammy se consulta desde PostgreSQL.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
import pandas as pd
from IPython.display import display
from src.common import connect, write_json
from src.profile import profile_frame
s=pd.read_csv(ROOT/'sources/spotify_dataset.csv')
with connect('grammy_source') as conn:
    cur=conn.execute('SELECT * FROM source.grammy_awards ORDER BY source_row_id')
    g=pd.DataFrame(cur.fetchall(),columns=[c.name for c in cur.description])
print('Spotify:',s.shape,'Grammy DB:',g.shape)

Spotify: (114000, 21) Grammy DB: (4810, 11)


## Estructura, completitud y claves candidatas
Los blancos Grammy se preservan como cadenas vacías en el origen relacional; se contabilizan explícitamente junto a los NULL.

In [2]:
for name,df in [('Spotify',s),('Grammy DB',g)]:
    missing=df.isna() | df.astype(str).apply(lambda c:c.str.strip().eq(''))
    print(name)
    display(pd.DataFrame({'tipo':df.dtypes.astype(str),'nulos_o_blancos':missing.sum(),'porcentaje':100*missing.mean(),'distintos':df.nunique()}))
    print('Duplicados exactos:',df.duplicated().sum())
print('Spotify ID repetido:',s.track_id.duplicated().sum())
print('Grammy ID repetido:',g.source_row_id.duplicated().sum())

Spotify


,tipo,nulos_o_blancos,porcentaje,distintos
Unnamed: 0,int64,0,0.000000,114000
track_id,object,0,0.000000,89741
artists,object,1,0.000877,31437
album_name,object,1,0.000877,46589
track_name,object,1,0.000877,73608
popularity,int64,0,0.000000,101
duration_ms,int64,0,0.000000,50697
explicit,bool,0,0.000000,2
danceability,float64,0,0.000000,1174
energy,float64,0,0.000000,2083


Duplicados exactos: 0
Grammy DB


,tipo,nulos_o_blancos,porcentaje,distintos
source_row_id,int64,0,0.000000,4810
year,object,0,0.000000,62
title,object,0,0.000000,62
published_at,object,0,0.000000,4
updated_at,object,0,0.000000,10
category,object,0,0.000000,638
nominee,object,6,0.124740,4132
artist,object,1840,38.253638,1659
workers,object,2190,45.530146,2367
img,object,1367,28.419958,1464


Duplicados exactos: 0
Spotify ID repetido: 24259
Grammy ID repetido: 0


## Categorías y variaciones de representación

In [3]:
display(g.category.value_counts().head(20).to_frame('registros'))
display(s.track_genre.value_counts().head(15).to_frame('registros'))
display(g.winner.value_counts(dropna=False).to_frame('registros'))
for name,series in [('Grammy categoría',g.category),('Grammy artista',g.artist),('Spotify artistas',s.artists)]:
    print(name,'originales=',series.nunique(),'casefold/trim=',series.str.strip().str.casefold().nunique())

,registros
category,
Song Of The Year,70
Record Of The Year,69
Album Of The Year,66
Best Opera Recording,64
Best Album Notes,63
Best Country Song,55
Best Instrumental Composition,55
Best New Artist,51
Best Historical Album,44


,registros
track_genre,
acoustic,1000
afrobeat,1000
alt-rock,1000
alternative,1000
ambient,1000
anime,1000
black-metal,1000
bluegrass,1000
blues,1000


,registros
winner,
True,4810


Grammy categoría originales= 638 casefold/trim= 630
Grammy artista originales= 1659 casefold/trim= 1657
Spotify artistas originales= 31437 casefold/trim= 31428


## Números, unidades y contradicciones de track_id

In [4]:
display(s.select_dtypes(include='number').describe().T)
cols=['artists','track_name','popularity','energy','danceability']
conflict=s.groupby('track_id')[cols].nunique(dropna=False).gt(1).any(axis=1)
print('IDs contradictorios:',conflict.sum())
display(s[s.track_id.isin(conflict[conflict].index)].head(12))
print('Duración cero (ms):',s.duration_ms.eq(0).sum())

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,114000.0,56999.500000,32909.109681,0.000,28499.75000,56999.500000,85499.2500,113999.000
popularity,114000.0,33.238535,22.305078,0.000,17.00000,35.000000,50.0000,100.000
duration_ms,114000.0,228029.153114,107297.712645,0.000,174066.00000,212906.000000,261506.0000,5237295.000
danceability,114000.0,0.566800,0.173542,0.000,0.45600,0.580000,0.6950,0.985
energy,114000.0,0.641383,0.251529,0.000,0.47200,0.685000,0.8540,1.000
key,114000.0,5.309140,3.559987,0.000,2.00000,5.000000,8.0000,11.000
loudness,114000.0,-8.258960,5.029337,-49.531,-10.01300,-7.004000,-5.0030,4.532
mode,114000.0,0.637553,0.480709,0.000,0.00000,1.000000,1.0000,1.000
speechiness,114000.0,0.084652,0.105732,0.000,0.03590,0.048900,0.0845,0.965
acousticness,114000.0,0.314910,0.332523,0.000,0.01690,0.169000,0.5980,0.996


IDs contradictorios: 720


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
29,29,2qESE1ZeWly7I3YjyTXmXh,Jason Mraz,Perfect Christmas Hits,Winter Wonderland,0,131760,False,0.620,0.309,...,-9.209,1,0.0495,0.788000,0.000000,0.1460,0.664,145.363,4,acoustic
130,130,72R0X0h8YaxYNpegeoOl0M,Parachute,The Way It Was,Kiss Me Slowly,61,235813,False,0.470,0.801,...,-6.426,1,0.0352,0.075800,0.000035,0.2180,0.431,173.920,4,acoustic
1013,1013,3BGxXQJWeLAEcwNKRFUxYA,BaianaSystem;BNegão,ATO 1: Navio Pirata,Reza Forte,38,232826,False,0.844,0.915,...,-4.060,0,0.1290,0.393000,0.002380,0.0847,0.872,145.997,4,afrobeat
1039,1039,2zQt5C0AIv27RhfJCRZdZ4,BaianaSystem,Duas Cidades,"Jah Jah Revolta, Pt. 2",39,309493,False,0.776,0.800,...,-4.704,0,0.0438,0.006310,0.073100,0.3500,0.583,143.989,4,afrobeat
1096,1096,3MZm4t1kRJ2e5JyaYQpjrF,Jackie Mittoo,The Keyboard King at Studio One,Black Organ,34,191506,False,0.837,0.598,...,-9.047,0,0.1810,0.391000,0.238000,0.0598,0.983,145.219,4,afrobeat
2007,2007,0y1QJc3SJVPKJ1OvFmFqe6,KALEO,A/B,Way down We Go,78,213706,False,0.489,0.505,...,-8.022,0,0.1170,0.579000,0.000333,0.1040,0.337,163.255,4,alt-rock
2012,2012,7uCBGNn991AM68fR4tSDco,The Score,Carry On,Born For This,69,237375,False,0.411,0.675,...,-6.251,0,0.0591,0.063000,0.000000,0.0920,0.545,130.134,4,alt-rock
2014,2014,75fT19n6ZQHr67Mkh86qfD,Red Hot Chili Peppers,Rock Brandneu,Tippa My Tongue,2,260808,False,0.573,0.913,...,-6.520,0,0.0409,0.008610,0.027300,0.1140,0.795,88.418,4,alt-rock
2018,2018,1yZWmKmLfTrFCxHvhFsdoN,The Killers;Phoebe Bridgers,The Greatest Rock Tunes 2022,Runaway Horses - Abridged,0,204466,False,0.499,0.223,...,-15.253,1,0.0338,0.912000,0.000000,0.0962,0.304,129.291,5,alt-rock
2023,2023,1bOENKlPYEDXyH9kLFUsnj,Nickelback,Timeless Rock Hits,Animals,3,185906,False,0.569,0.963,...,-3.069,1,0.0593,0.000787,0.025600,0.2200,0.820,76.995,4,alt-rock


Duración cero (ms): 1


## Cobertura temporal
El año Grammy se conserva tal como lo entrega la fuente. Las fechas published_at y updated_at describen publicación editorial. Spotify no proporciona fecha de snapshot ni fecha de lanzamiento.

In [5]:
years=pd.to_numeric(g.year,errors='coerce')
print('Año inválido:',years.isna().sum(),'rango:',years.min(),years.max())
print('Huecos:',sorted(set(range(int(years.min()),int(years.max())+1))-set(years)))
for c in ['published_at','updated_at']:
    values=pd.to_datetime(g[c],errors='coerce',utc=True)
    print(c,'no parseables=',values.isna().sum(),'mínimo=',values.min(),'máximo=',values.max())

Año inválido: 0 rango: 1958 2019
Huecos: []
published_at no parseables= 0 mínimo= 2017-11-28 08:03:45+00:00 máximo= 2020-05-19 12:10:28+00:00
updated_at no parseables= 0 mínimo= 2017-11-28 08:03:45+00:00 máximo= 2020-09-01 19:16:40+00:00


## Compatibilidad entre fuentes y evidencia del cruce
La siguiente medición aplica el contrato documentado sin modificar las fuentes. Los resultados justifican las advertencias de cobertura y la validación preparada.

In [6]:
from src.transform import integrate_frames
prepared,quarantine,reconciliation=integrate_frames(s,g)
display(pd.Series(reconciliation,dtype=object).to_frame('valor'))
display(pd.crosstab(prepared.category,prepared.match_status).loc[lambda x:x.sum(axis=1)>0].head(20))
write_json(ROOT/'docs/evidence/notebook_profile.json',{'spotify':profile_frame(s),'grammy_db':profile_frame(g),'integration':reconciliation})

,valor
spotify_raw_rows,114000
spotify_unique_ids,89741
conflicting_track_ids,720
quarantined_spotify_rows,1933
spotify_canonical_tracks,89020
grammy_raw_rows,4810
prepared_rows,4810
status_counts,"{'out_of_scope': 4649, 'unmatched': 115, 'matc..."
eligible_records,161
matched_records,33


match_status,ambiguous,matched,out_of_scope,unmatched
category,,,,
Album Of Best Original Score Written For A Motion Picture Or A Television Special,0,0,3,0
Album Of Best Original Score Written For A Motion Picture Or Television Special,0,0,1,0
Album Of The Year,0,0,66,0
Album Of The Year (Other Than Classical),0,0,3,0
Album Of The Year - Classical,0,0,3,0
Album Of The Year -- Classical,0,0,2,0
"Album Of The Year, Classical",0,0,8,0
Best Accompaniment Arrangement For Vocalist(s) Or Instrumentalist(s),0,0,1,0
Best Album Cover,0,0,8,0


'/opt/airflow/docs/evidence/notebook_profile.json'

## Interpretación y decisiones
Véanse docs/02_profiling_and_risks.md y docs/04_quality_rules.md. El flag winner constante no permite distinguir ganadores; no se usa como KPI. Los conflictos por ID y las coincidencias ambiguas no se resuelven arbitrariamente. La ausencia de coincidencia no demuestra ausencia de reconocimiento.